In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

import deepchem as dc

### Load HIV dataset, clean smiles, save it ... 

In [2]:
%%capture
_, datasets, _ = dc.molnet.load_hiv()
ds_train, ds_valid, ds_test = datasets

In [3]:
def undo_BrCl_singles(smi):
    smi = smi.replace('R','Br')
    return smi.replace('L','Cl')
def do_BrCl_singles(smi):
    smi = smi.replace('Br','R')
    return smi.replace('Cl','L')      

In [30]:
from rdkit import RDLogger  
RDLogger.DisableLog('rdApp.*')
from utilities.rdkit_utils import *
from tqdm.notebook import tqdm

tokens_salsa = '#%()+-0123456789<=>BCFHILNOPRSX[]cnos'
tokens_hiv = 'Ub)gsP5u=8H[d]noW06%2GLlO+NFSTa9ZCeI-r4(1ihAVBRc3M7pt#'

cuts = ['train', 'test'] #,'val','test']

# Get canon smiles

bad_smis = []
all_smis = []
raw_smis = []

for i,cut in enumerate(cuts):
    
    ds = datasets[i]
    X_and_ys = zip(ds.ids, ds.y)
    
    smis = []
    labs = []
    
    for smi,lab in tqdm(X_and_ys, total=len(ds)):        
        _smi = get_cansmiles(smi)
        components = _smi.split('.')
        components = sorted(components, key=lambda x: len(x))
        smi = components[-1]
        
        if smi:
            raw_smis.append(smi)
            
            # Need to match SALSA's vocab !!!
            smi = do_BrCl_singles(smi)  
            
            if not all(c in tokens_salsa for c in smi):
                bad_smis.append(smi)
                if 'U' in smi:
                    print('---\n',smi)
                    
            elif len(smi) > 120:
                bad_smis.append(smi)
            else:
                smis.append(smi)
                all_smis.append(smi)
                labs.append(lab)
            
    df = pd.DataFrame(smis,columns=['smiles'])
    df.to_csv(f'data/model_ready/hiv/{cut}/anchor_smiles.csv',index=False)
    
    df = pd.DataFrame(labs,columns=['y'])
    df.to_csv(f'data/model_ready/hiv/{cut}/labels.csv',index=False)

  0%|          | 0/32901 [00:00<?, ?it/s]

  0%|          | 0/4113 [00:00<?, ?it/s]

---
 COc1cccc2c1[OH+][U]134(=O)(=O)(O)(O)([O+]=C(c5ccncc5)[N-][N+]1=C2)[O+]=C(c1ccncc1)[N-][N+]3=Cc1cccc(OC)c1[OH+]4
---
 COc1cccc2c1[OH+][U-5](=O)(=O)(O)(O)[O+]=C2
---
 COc1cccc2c1[OH+][U+2]1(=O)(=O)(O)(O)(O)[O+]=C(N)[N-][N+]1=C2


In [31]:
len(all_smis)

35611

### Look at the HIV tokens not in SALSA vocab ...

In [32]:
print(len(bad_smis), len(all_smis), len(raw_smis))

v = list(set(''.join(all_smis)))
print(''.join(v))

1403 35611 37014
0LI4[(2cF=BHs396-OP#)no8R57%]+CS1N


In [33]:
len(raw_smis)-len(bad_smis)

35611

In [34]:
bad_tokens = list(set(list(tokens_hiv)) - set(tokens_salsa))
# bad_tokens = list(set(list(tokens_salsa)) ^ set(tokens_hiv))
print(len(bad_tokens))
print(bad_tokens)

20
['p', 'U', 'e', 'a', 'g', 'V', 'G', 'd', 't', 'i', 'l', 'Z', 'b', 'W', 'h', 'u', 'M', 'r', 'T', 'A']


### Load back in the HIV dataset.

In [39]:
df_hiv = pd.read_csv('data/model_ready/hiv/train/anchor_smiles.csv')
hiv_vocab = list(set(list(''.join(df_hiv.smiles.values))))
print(len(hiv_vocab))
print(len(tokens_salsa))

34
37


### Load models and get latents ... 

In [45]:
import torch.nn as nn
import torch
from seqAE_model import SeqAutoencoder
from contra_seq_dataset import *
from torch.utils.data import DataLoader, RandomSampler
from tqdm import tqdm

def get_latents(model_tag, ds_cut, load_bs=32):
    tag = model_tag
    cut = ds_cut
    bs = load_bs

    n_epochs = 30
    use_cuda = True
    empty_cuda = True
    cuda_ids = [0,1,2,3]
    model = SeqAutoencoder(dim_emb=512, heads=8, dim_hidden=32,
                           L_enc=6, L_dec=6, dim_ff=2048, 
                           drpt=0.1, actv='relu', eps=0.6, b_first=True)

    p = f'/home/kat/Repos/SALSA/results/models/{tag}/{n_epochs-1:02}.pt'
    model.load_state_dict(torch.load(p), strict = False)
    if empty_cuda:
        torch.cuda.empty_cache()
    if use_cuda:
        if len(cuda_ids) == 1:
            cuda_id = cuda_ids[0]
            device = torch.device(f"cuda:{cuda_id}")
        elif len(cuda_ids) > 1:
            device =  torch.device("cuda")
            print(f"Using {len(cuda_ids)} GPUs!")
            model = nn.DataParallel(model, device_ids=cuda_ids)
            model.to(device)
    else:
        device = torch.device("cpu")
        model = model.to(device)
    model = model.eval()
    print(f"Loaded model weights from {p}!")

    p = f'data/model_ready/hiv/{cut}/anchor_smiles.csv'
    ds = ContraSeqDataset(p)
    df = get_dataset_array(p)

    loader = DataLoader(ds, batch_size=bs, sampler=range(len(df)), 
                        num_workers=0, pin_memory=True)
    latents = []
    for samp in tqdm(loader, total=len(df)//bs):
        for k,v in samp.items():
            if torch.is_tensor(v):
                samp[k] = v.to(device)
        latent, _ = model.forward(samp['seq'], samp['pad_mask'], 
                                  samp['avg_mask'], samp['out_mask'])
        latent = latent.cpu().detach().numpy()
        latents.append(latent)
    latents = np.concatenate(latents, axis=0)
    
    return latents

In [46]:
# # # # # # # # # # # #
tag = '2022041804_04' 
load_bs = 32
# # # # # # # # # # # #

X_train = get_latents(tag, 'train', load_bs) 
X_test = get_latents(tag, 'test', load_bs) 

Using 4 GPUs!
Loaded model weights from /home/kat/Repos/SALSA/results/models/2022041804_04/29.pt!


995it [02:18,  7.19it/s]                         


Using 4 GPUs!
Loaded model weights from /home/kat/Repos/SALSA/results/models/2022041804_04/29.pt!


119it [00:16,  7.21it/s]                         


In [ ]:
y_train = pd.read_csv('data/model_ready/hiv/train/labels.csv')['y'].values
y_test = pd.read_csv('data/model_ready/hiv/test/labels.csv')['y'].values

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

### Run single random forest ... 

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier as BalRF
from sklearn.model_selection import StratifiedKFold

seedy = 666

skf = StratifiedKFold(n_splits=5)
skf.get_n_splits(X, y)

clf = BalRF(n_estimators=100, max_features="auto",sampling_strategy='auto',
            random_state=seedy,n_jobs=-1,oob_score=True)

df_y = pd.DataFrame(y_binary, columns=['y_test'])

oobs = []
for i,(train_idx, test_idx) in enumerate(skf.split(X,y)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    clf.fit(X_train,y_train)
    print(f'OOB, split {i}: {clf.oob_score_}')

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)
    df_y.loc[test_idx,'y_prob'] = y_prob[:,1]
    df_y.loc[test_idx,'y_pred'] = y_pred
    oobs.append(clf.oob_score_)
    
print(f"Mean OOB: {sum(oobs)/5}")

In [47]:
# from contra_seq_dataset import *
# from torch.utils.data import DataLoader, RandomSampler

# p = 'data/model_ready/hiv/train/anchor_smiles.csv'

# ds = ContraSeqDataset(p)
# df = get_dataset_array(p)

# bs = 32

# from tqdm.notebook import tqdm
# loader = DataLoader(ds, batch_size=bs, sampler=range(len(df)), 
#                     num_workers=0, pin_memory=True)
# latents = []
# for samp in tqdm(loader, total=len(df)//bs):
#     for k,v in samp.items():
#         if torch.is_tensor(v):
#             samp[k] = v.to(device)
#     latent, _ = model.forward(samp['seq'], samp['pad_mask'], 
#                               samp['avg_mask'], samp['out_mask'])
#     latent = latent.cpu().detach().numpy()
#     latents.append(latent)
# latents = np.concatenate(latents, axis=0)

In [ ]:
# from imblearn.ensemble import BalancedRandomForestClassifier as BalRF
# from sklearn.model_selection import StratifiedKFold

# seedy = 666

# skf = StratifiedKFold(n_splits=5)
# skf.get_n_splits(X, y)

# clf = BalRF(n_estimators=100, max_features="auto",sampling_strategy='auto',
#             random_state=seedy,n_jobs=-1,oob_score=True)

# df_y = pd.DataFrame(y_binary, columns=['y_test'])

# oobs = []
# for i,(train_idx, test_idx) in enumerate(skf.split(X,y)):
#     X_train, X_test = X[train_idx], X[test_idx]
#     y_train, y_test = y[train_idx], y[test_idx]

#     clf.fit(X_train,y_train)
#     print(f'OOB, split {i}: {clf.oob_score_}')

#     y_pred = clf.predict(X_test)
#     y_prob = clf.predict_proba(X_test)
#     df_y.loc[test_idx,'y_prob'] = y_prob[:,1]
#     df_y.loc[test_idx,'y_pred'] = y_pred
#     oobs.append(clf.oob_score_)
    
# print(f"Mean OOB: {sum(oobs)/5}")